> **Public release copy.** Set `<DATA_DIR>` in the CONFIG cell to the folder holding the large input files (see README / DATA availability). Internal validation cells have been removed. The small curation TSVs are included in this folder.


# 04 -- Portal artifacts (interactive heat-tree JSON + portal TSV/fasta)

**Stage 4 of the streamlined RNAquarium virus pipeline.** Produces the web-portal artifacts: the interactive radial heat-tree JSON and the portal
curated TSV + fasta.
**Both are produced from the SAME curated object (01's outputs)** so they can never disagree
-- so the portal artifacts and the figures stay consistent.

**Inputs**
- `curated_contigs.tsv`   (from 01 -- the curated set; this IS the portal TSV)
- `curated_clusters.tsv`  (from 01 -- per-virus name, Category, classification, counts)
- `curated.fasta`         (from 01 -- already in portal header format)

**Outputs (tagged):**
- `RNAquarium_viruses_fishassociated_curated_<date>.tsv`   -> portal TSV  (= curated_contigs)
- `RNAquarium_viruses_fishassociated_curated_<date>.fasta` -> portal fasta (= curated.fasta)
- `heattree_curated_interactive.json`                       -> portal interactive heat-tree

The curated set is taken directly from step 01 (the single source of truth); this step just
formats and emits the portal artifacts.


## CONFIG -- edit paths here

In [ ]:
library(tidyverse)
library(metacoder)
library(jsonlite)

# -- CONFIG -- edit paths here -----------------------------------------------
workingpath <- getwd()

curated_contigs_file  <- "curated_contigs.tsv"    # from 01
curated_clusters_file <- "curated_clusters.tsv"   # from 01
curated_fasta_file    <- "curated.fasta"          # from 01

today <- Sys.Date()
out_portal_tsv   <- paste0("RNAquarium_viruses_fishassociated_curated_", today, ".tsv")
out_portal_fasta <- paste0("RNAquarium_viruses_fishassociated_curated_", today, ".fasta")
out_portal_json  <- "heattree_curated_interactive.json"

genus_level_curated <- c("Chuvivirus", "Deltainfluenzavirus")
R_radius <- 160   # radial layout radius per depth
# ----------------------------------------------------------------------------

## 1. Portal TSV + fasta (the F outputs -- now just copies of 01's curated set)

01 already produced the curated contig table and a portal-formatted fasta. The portal TSV
*is* `curated_contigs.tsv`; we just write it under the dated portal name. The fasta is copied
through. No re-derivation.

In [ ]:
curated  <- read_tsv(file.path(workingpath, curated_contigs_file),  show_col_types = FALSE)
clusters <- read_tsv(file.path(workingpath, curated_clusters_file), show_col_types = FALSE)
cat("Curated contigs:", nrow(curated), "| viruses:", n_distinct(curated$clusterLCA_curated), "\n")

# -> portal TSV
write_tsv(curated, file.path(workingpath, out_portal_tsv))
cat("Wrote portal TSV:", out_portal_tsv, "\n")

# -> portal fasta (copy 01's curated.fasta through under the dated name)
if (file.exists(file.path(workingpath, curated_fasta_file))) {
  file.copy(file.path(workingpath, curated_fasta_file),
            file.path(workingpath, out_portal_fasta), overwrite = TRUE)
  cat("Wrote portal fasta:", out_portal_fasta, "\n")
} else {
  cat("NOTE:", curated_fasta_file, "not found -- run 01 first.\n")
}

## 2. Build the metacoder tree for the JSON (from precomputed classification)

Parse the `classification` column from `curated_contigs.tsv` (no taxize API call).
Then compute per-taxon abundance for the node metrics the portal shows.

In [ ]:
tree_input <- curated %>% dplyr::filter(!is.na(classification), classification != "")
obj <- parse_tax_data(tree_input, class_cols = "classification", class_sep = ";")

metric_cols <- intersect(c("rel_abundance", "bits_NTorNR", "length"), names(obj$data$tax_data))
obj$data$tax_abund <- calc_taxon_abund(obj, "tax_data", cols = metric_cols)
cat("Parsed tree:", length(obj$taxon_ids()), "taxa\n")

## 3. Radial layout (BFS depths + leaf-count angle allocation)

Hand-rolled radial layout: compute depths via BFS, leaf counts bottom-up, then allocate angle
to each node proportional to its leaf count. Produces x/y for each taxon.

In [ ]:
edges_df <- obj$edge_list %>% as_tibble() %>% dplyr::filter(!is.na(from), !is.na(to))
ids   <- obj$taxon_ids(); nms <- obj$taxon_names()
child_map <- split(edges_df$to, edges_df$from)

root_candidates <- setdiff(ids, unique(edges_df$to))
root_id <- if (length(root_candidates) == 1) root_candidates[1] else
           { rp <- intersect(root_candidates, unique(edges_df$from)); if (length(rp)) rp[1] else root_candidates[1] }

# BFS depths
node_depth <- setNames(rep(NA_integer_, length(ids)), ids); node_depth[root_id] <- 0L
queue <- root_id
while (length(queue)) { cur <- queue[1]; queue <- queue[-1]
  kids <- child_map[[cur]]; if (!is.null(kids)) for (k in kids) { node_depth[k] <- node_depth[cur]+1L; queue <- c(queue,k) } }

# leaf counts (bottom-up)
leaf_counts <- setNames(rep(0L, length(ids)), ids)
parent_map <- setNames(rep(NA_character_, length(ids)), ids)
for (i in seq_len(nrow(edges_df))) parent_map[edges_df$to[i]] <- edges_df$from[i]
for (tid in ids) if (is.null(child_map[[tid]])) leaf_counts[tid] <- 1L
proc <- names(sort(node_depth, decreasing = TRUE, na.last = NA))
for (tid in proc) { kids <- child_map[[tid]]
  if (!is.null(kids) && length(kids)) leaf_counts[tid] <- sum(leaf_counts[kids], na.rm = TRUE)
  if (leaf_counts[tid] == 0L && is.null(child_map[[tid]])) leaf_counts[tid] <- 1L }

# radial coords
coords <- data.frame(taxon_id = ids, x = 0, y = 0, stringsAsFactors = FALSE); rownames(coords) <- ids
lay <- function(nid, aS, aE, d) {
  a <- (aS+aE)/2; r <- d*R_radius
  coords[nid,"x"] <<- r*cos(a); coords[nid,"y"] <<- r*sin(a)
  kids <- child_map[[nid]]; if (is.null(kids) || !length(kids)) return()
  tl <- leaf_counts[nid]; cur <- aS
  for (k in kids) { sp <- (leaf_counts[k]/tl)*(aE-aS); lay(k, cur, cur+sp, d+1); cur <- cur+sp }
}
lay(root_id, -pi, pi, 0)
cat("Radial layout done; root =", nms[match(root_id, ids)], "| leaves =", leaf_counts[root_id], "\n")

## 4. Assemble node/edge tables + write JSON

Attach rank (ICTV suffix rules + curated Category), abundance, coords, per-virus counts, then
write nodes+edges+metadata to the portal JSON.

In [ ]:
assign_rank <- function(name) {
  if (is.na(name) || name=="") return(NA_character_)
  if (name=="Viruses") return("superkingdom")
  if (str_detect(name,"idae$")) return("family"); if (str_detect(name,"ales$")) return("order")
  if (str_detect(name,"cetes$")) return("class");  if (str_detect(name,"cota$")) return("phylum")
  if (str_detect(name,"virae$")) return("kingdom");if (str_detect(name,"viria$")) return("realm")
  NA_character_
}
get_cls <- function(tid){ chain<-character(0); cur<-tid
  while(!is.na(cur)){ idx<-match(cur,ids); if(!is.na(idx)) chain<-c(nms[idx],chain); cur<-parent_map[cur] }
  paste(chain,collapse=";") }
classifications <- sapply(ids, get_cls)

curated_cat <- clusters %>% select(clusterLCA_curated, Category) %>% distinct(clusterLCA_curated,.keep_all=TRUE)

node_data <- tibble(taxon_id=ids, name=nms, classification=classifications) %>%
  mutate(depth = str_count(classification,";")+1,
         is_leaf = !taxon_id %in% edges_df$from,
         rank_suffix = sapply(name, assign_rank)) %>%
  left_join(curated_cat, by=c("name"="clusterLCA_curated")) %>%
  mutate(rank = case_when(
    !is.na(Category) & name %in% genus_level_curated ~ "genus",
    !is.na(Category) ~ "species",
    !is.na(rank_suffix) ~ rank_suffix,
    depth>=9 ~ "species", depth==8 ~ "genus", depth==1 ~ "superkingdom",
    depth==2 ~ "realm", depth==3 ~ "kingdom", depth==4 ~ "phylum",
    depth==5 ~ "class", depth==6 ~ "order", depth==7 ~ "family",
    TRUE ~ paste0("level_", depth))) %>%
  select(-rank_suffix) %>%
  left_join(as_tibble(obj$data$tax_abund), by="taxon_id") %>%
  left_join(coords, by="taxon_id")

# --- n_transcripts: seed curated nodes with contigcount, then PROPAGATE UP the tree ---
# (internal nodes sum their children, in reverse-depth order)
tc <- setNames(rep(0L, length(ids)), ids)
metrics <- clusters %>% select(clusterLCA_curated, contigcount, bioprojectcount)
for (i in seq_len(nrow(metrics))) {
  idx <- which(nms == metrics$clusterLCA_curated[i])
  if (length(idx) > 0) tc[ids[idx[1]]] <- as.integer(metrics$contigcount[i])
}
# proc is reverse-depth order (defined earlier); sum children up to parents
for (tid in proc) {
  kids <- child_map[[tid]]
  if (!is.null(kids) && length(kids) > 0) tc[tid] <- tc[tid] + sum(tc[kids], na.rm = TRUE)
}
node_data <- node_data %>%
  left_join(tibble(taxon_id = names(tc), n_transcripts = as.integer(tc)), by = "taxon_id") %>%
  mutate(n_transcripts = tidyr::replace_na(n_transcripts, 0L)) %>%
  # n_bioprojects: leaf-only (no propagation), joined by curated name
  left_join(metrics %>% select(name = clusterLCA_curated, n_bioprojects = bioprojectcount), by = "name")

avail_num <- intersect(c("rel_abundance","bits_NTorNR","length"), names(node_data))
nodes_export <- node_data %>%
  select(taxon_id,name,classification,depth,rank,is_leaf,x,y,
         n_transcripts,n_bioprojects,any_of(avail_num),Category) %>%
  mutate(across(where(is.numeric), ~round(.x,4)))
edges_export <- edges_df %>% rename(source=from, target=to)

# richer metadata block (parity with the old portal JSON)
tree_json <- list(nodes=nodes_export, edges=edges_export,
  metadata=list(
    n_taxa     = n_distinct(nodes_export$taxon_id),
    n_families = sum(nodes_export$rank == "family", na.rm = TRUE),
    n_genera   = sum(nodes_export$rank == "genus",  na.rm = TRUE),
    n_species  = sum(nodes_export$rank == "species",na.rm = TRUE),
    n_leaves   = sum(nodes_export$is_leaf),
    n_curated  = sum(!is.na(nodes_export$Category)),
    date_exported = as.character(today),
    numeric_columns = c(avail_num, "n_transcripts", "n_bioprojects"),
    layout = "radial", radius_per_depth = R_radius))

# -> portal JSON
write_json(tree_json, file.path(workingpath, out_portal_json), pretty=TRUE, auto_unbox=TRUE)
cat("Wrote portal JSON:", out_portal_json, "| nodes:", nrow(nodes_export),
    "| edges:", nrow(edges_export), "| curated:", sum(!is.na(nodes_export$Category)), "\n")

## 5. Notes

- Portal TSV/fasta are now byte-identical to 01's curated outputs (just dated names) -- no
  re-derivation, so they cannot drift from the figures.